# Esquemas propios

### Importación de librerías

In [1]:
from pathlib import Path
import os
import subprocess
import re
import mysql.connector
import sqlparse
from dotenv import load_dotenv

### Rutas

In [2]:
contrasenia = '../.env'

### Ruta de directorio

In [4]:
ruta_proyecto = Path.cwd().resolve()

if ruta_proyecto.name == 'notebooks':
    ruta_proyecto = ruta_proyecto.parent

ruta_esquemas = ruta_proyecto / 'esquemas'

### Archivos de esquemas

In [5]:
esquema_1 = ruta_esquemas / 'sales_clothes.sql'
esquema_2 = ruta_esquemas / 'tenis_mesa.sql'

### Conexión de la base de datos

In [6]:
# Contraseña del proyecto
load_dotenv(contrasenia)

# Conexión con MySQL
conexion = mysql.connector.connect(host='mysql', port=3306, user='root',
                                   password=os.getenv('MYSQL_ROOT_PASSWORD'))

# Verificar la conexión
if conexion.is_connected():
    print('Conexión exitosa')
    cursor = conexion.cursor()

Conexión exitosa


### Creación de base de datos

In [7]:
try:
    for archivo in [esquema_1, esquema_2]:

        # Nombre de la base a partir del archivo.
        nombre_bd = archivo.stem.lower()

        # Verificar si la base ya existe.
        cursor.execute(
            """
            SELECT SCHEMA_NAME
            FROM INFORMATION_SCHEMA.SCHEMATA
            WHERE SCHEMA_NAME = %s
            """,
            (nombre_bd,)
        )

        if cursor.fetchone():
            print(f"La base '{nombre_bd}' ya existe. Se omite.")
            continue

        # Leer instrucciones SQL.
        contenido_sql = archivo.read_text( encoding="utf-8-sig")

        # Crear la base de datos.
        cursor.execute(f"CREATE DATABASE `{nombre_bd}`")

        # Seleccionar la base.
        cursor.execute(f"USE `{nombre_bd}`")

        # Ejecutar las instrucciones del archivo.
        cursor.execute(contenido_sql)

        # Procesar los resultados de cada instrucción.
        while cursor.nextset():
            pass

        # Confirmar transacciones pendientes.
        conexion.commit()

        print(f"Base '{nombre_bd}' creada correctamente.")

finally:
    pass

La base 'sales_clothes' ya existe. Se omite.
La base 'tenis_mesa' ya existe. Se omite.


### Cerrando conexión

In [10]:
cursor.close()
conexion.close()

### Esquemas datos

In [11]:
esquema_dato_1 = ruta_esquemas / 'sales_clothes_datos.sql'
esquema_dato_2 = ruta_esquemas / 'tenis_mesa_datos.sql'
esquemas_datos = [esquema_dato_1, esquema_dato_2]

### Insertando datos a las bases de datos de esquemas propios

In [12]:
for esquema in esquemas_datos:

    esquema = Path(esquema)

    # Obtener nombre de la base de datos.
    nombre_bd = esquema.stem.removesuffix("_datos")

    # Leer el archivo SQL.
    with open(esquema, "r", encoding="utf-8") as archivo:
        contenido = archivo.read()

    # Separar las instrucciones y eliminar comentarios.
    instrucciones = sqlparse.split(sqlparse.format(contenido, strip_comments=True))

    # Conservar únicamente instrucciones INSERT INTO.
    inserciones = [instruccion.strip() for instruccion in instrucciones if re.match(r"^\s*INSERT\s+INTO\b", instruccion, re.I)]

    if not inserciones:
        raise ValueError(f"No hay instrucciones INSERT en {esquema.name}")

    # Identificar las tablas que recibirán datos.
    tablas = set()

    for instruccion in inserciones:
        coincidencia = re.match(r"INSERT\s+INTO\s+`?([A-Za-z_][A-Za-z0-9_]*)`?", instruccion, re.I)

        if not coincidencia:
            raise ValueError(f"No se pudo identificar la tabla: {instruccion[:100]}")

        tablas.add(coincidencia.group(1))

    # Conectar a la base de datos existente.
    conexion = mysql.connector.connect(host=os.getenv("MYSQL_HOST", "mysql"),
                                       port=int(os.getenv("MYSQL_PORT", 3306)),
                                       user=os.getenv("MYSQL_USER", "root"),
                                       password=os.getenv("MYSQL_PASSWORD") or os.getenv("MYSQL_ROOT_PASSWORD"),
                                       database=nombre_bd,
                                       charset="utf8mb4")
    cursor = conexion.cursor()

    try:
        # Eliminar registros anteriores.
        for tabla in sorted(tablas):
            cursor.execute(f"DELETE FROM `{tabla}`")

        # Insertar los nuevos registros.
        for instruccion in inserciones:
            cursor.execute(instruccion)

        # Guardar los cambios.
        conexion.commit()

        print(f"Base de datos: {nombre_bd}")
        print(f"Tablas actualizadas: {len(tablas)}")
        print("Inserciones completadas correctamente.\n")

    except Exception:
        conexion.rollback()
        raise

    finally:
        cursor.close()
        conexion.close()

Base de datos: sales_clothes
Tablas actualizadas: 5
Inserciones completadas correctamente.

Base de datos: tenis_mesa
Tablas actualizadas: 6
Inserciones completadas correctamente.

